# Part 2: Consume KB as an MCP endpoint

In Part 1, you built a multi-source knowledge base with indexed HR and health documents. In this part, you'll learn how that knowledge base can be consumed as an **MCP (Model Context Protocol) endpoint** — meaning developer tools and agents can query it using the standard MCP protocol.

Azure AI Search automatically exposes every knowledge base as an MCP server. You don't need to deploy anything extra — just configure your MCP client to point at the endpoint.

## What is MCP?

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open standard that lets AI models and developer tools connect to external data sources and tools through a unified interface. When your knowledge base is exposed as an MCP endpoint, any MCP-compatible client can retrieve grounded answers from it.

## Step 1: Build the MCP endpoint URL

Every knowledge base in Azure AI Search has an MCP endpoint at this URL:

```
https://<search-service>.search.windows.net/knowledgebases/<kb-name>/mcp?api-version=2025-11-01-preview
```

Run the cell below to construct your knowledge base's MCP URL.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
api_key = os.environ["AZURE_SEARCH_ADMIN_KEY"]
knowledge_base_name = "hr-and-health-docs-knowledge-base"

mcp_url = f"{endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version=2025-11-01-preview"

print("Your KB's MCP endpoint URL:")
print(mcp_url)

## Step 2: Test with the Responses API

The [Azure OpenAI Responses API](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/responses?view=foundry-classic#using-remote-mcp-servers) supports MCP tools directly. You can connect your knowledge base as an MCP tool and query it in a single call — no separate client configuration needed.

The cell below creates an OpenAI client pointing at your Azure OpenAI endpoint and calls `responses.create()` with your knowledge base connected as an MCP tool. Set `require_approval` to `never` so the model can retrieve from the knowledge base without an extra round-trip.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]
azure_openai_chatgpt_deployment = os.getenv("AZURE_OPENAI_CHATGPT_DEPLOYMENT", "gpt-4.1")

# Learn more at https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses
client = OpenAI(
    api_key=azure_openai_key,
    base_url=f"{azure_openai_endpoint.rstrip('/')}/openai/v1/",
)

response = client.responses.create(
    model=azure_openai_chatgpt_deployment,
    tool_choice="required",
    tools=[
        {
            "type": "mcp",
            "server_label": "zava-knowledge-base",
            "server_url": mcp_url,
            "headers": {
                "api-key": api_key
            },
            "require_approval": "never",
        },
    ],
    input="What health plans does Zava offer?",
)

print(response.output_text)

## Step 3: Configure GitHub Copilot CLI

[GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/install-copilot-cli) supports MCP servers via an HTTP configuration file. To connect your knowledge base:

1. Create or edit `~/.copilot/mcp-config.json`
2. Add an entry for your knowledge base MCP endpoint

The cell below generates the configuration file content. Copy it to `~/.copilot/mcp-config.json`.

In [ ]:
import json

mcp_config = {
    "mcpServers": {
        "zava-knowledge-base": {
            "type": "http",
            "url": mcp_url,
            "headers": {
                "api-key": api_key
            }
        }
    }
}

print(json.dumps(mcp_config, indent=2))

## Step 4: Test with GitHub Copilot CLI

Once you've saved the configuration file, you can query your knowledge base directly from the command line using GitHub Copilot CLI:

```bash
copilot -i "What health plans does Zava offer?"
```

Copilot CLI will route the question to your knowledge base MCP endpoint and return grounded answers based on your indexed HR and health documents.

> **📌 Prerequisites**
> - [Install GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/set-up-copilot-cli/install-copilot-cli)
> - A GitHub Copilot subscription (Individual, Business, or Enterprise)
> - The `~/.copilot/mcp-config.json` file configured as shown above

## Summary

You've seen how every Azure AI Search knowledge base is automatically available as an MCP endpoint. By adding a simple configuration to `~/.copilot/mcp-config.json`, developer tools like GitHub Copilot CLI can query your enterprise knowledge base using the standard MCP protocol.

**Key concepts:**
- Knowledge bases are automatically exposed as MCP endpoints — no extra deployment needed
- The MCP URL follows a predictable pattern based on your search service and KB name
- Authentication uses the same API key as the Search service
- Any MCP-compatible client can connect to the endpoint

➡️ Continue to [Part 3: Add a non-authenticated MCP source (Learn)](part3-mcp-unauthenticated.ipynb) to connect an external MCP server as a live knowledge source in your knowledge base.